In [12]:
import pandas as pd
from openai import OpenAI
from autoddg import AutoDDG
from autoddg.utils import get_sample
from autoddg.evaluation import BaseEvaluator
from typing import Optional
# --- Import custom files ---
from prompts import ALL_RELATED_WORK_PROMPTS
from utils import log_result, run_description_experiment
import os 
import json
from cache_utils import run_with_caching, load_profile_from_cache#, MockAutoDDG           #Mock is for testing



[SETUP] Project Root: /Users/sara/Desktop/NYUFALL2025/NLP/FinalProject/AutoDDG-Enhanced
[SETUP] Cache Directory: /Users/sara/Desktop/NYUFALL2025/NLP/FinalProject/AutoDDG-Enhanced/prompt-experiments/profile_cache


In [ ]:

import openai
# --- LLM Config ---
MODEL_CONFIG = {
    "base_url": "http://localhost:11434/v1",
    "api_key": "ollama",
    "model_name": "llama3.1:8b",
}

# Initialize Core Tools

client = OpenAI(api_key=MODEL_CONFIG["api_key"], base_url=MODEL_CONFIG["base_url"])


In [15]:
import json
import pandas as pd

results = pd.read_csv("results.csv")
DATABASE_PATH_ = "../src/autoddg/database.json"

with open(DATABASE_PATH_, "r", encoding="utf-8") as f:
    db = json.load(f)

# dataset_name -> description lookup
name_to_desc = {
    v["dataset_name"].strip(): v.get("description", "")
    for v in db.values()
    if "dataset_name" in v
}

# add column (no overwriting other columns)
results["Reference_Description"] = (
    results["Dataset_Name"].astype(str).str.strip().map(name_to_desc)
)

# optional: keep blanks instead of NaN
results["Reference_Description"] = results["Reference_Description"].fillna("")

results.to_csv("results_refdesc.csv", index=False)




In [16]:
import pandas as pd
from enhanced_eval import evaluate_all

# Step 1: load results file
df = pd.read_csv("results_refdesc.csv")

# Step 2: compute metrics for each row ( generation)
all_metrics = []
for _, row in df.iterrows():
    metrics = evaluate_all(row, client, MODEL_CONFIG["model_name"] )
    all_metrics.append(metrics)

# Step 3: convert to DataFrame
metrics_df = pd.DataFrame(all_metrics)

# Step 4: save full evaluation metrics
metrics_df.to_csv("enhanced_results.csv", index=False)

print("Enhanced evaluation saved to enhanced_results.csv")


Some weights of RobertaModel were not initialized from the model checkpoint at roberta-large and are newly initialized: ['pooler.dense.bias', 'pooler.dense.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.


TypeError: 'ChatCompletionMessage' object is not subscriptable